In [1]:
# ============================================================
# Vision AI Fundamentals: Fashion-MNIST Image Classifier
# Author: Shubham Mani Tripathi
# Description:
# Build, train, compare, and evaluate ANN, Basic CNN, and Deeper CNN
# models on Fashion-MNIST image classification dataset.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# -----------------------------
# 1. Project Config
# -----------------------------
RANDOM_STATE = 42
EPOCHS = 15
BATCH_SIZE = 64
MODEL_DIR = "models"
RESULT_DIR = "results"

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]


# -----------------------------
# 2. Load and Preprocess Dataset
# -----------------------------
def load_data():
    print("Loading Fashion-MNIST dataset...")

    (x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

    x_train = x_train / 255.0
    x_test = x_test / 255.0

    x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
    x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)

    y_train_cat = keras.utils.to_categorical(y_train, 10)
    y_test_cat = keras.utils.to_categorical(y_test, 10)

    print("Training images:", x_train.shape)
    print("Testing images:", x_test.shape)

    return x_train, y_train, y_train_cat, x_test, y_test, y_test_cat


# -----------------------------
# 3. Build Models
# -----------------------------
def build_ann_model():
    model = keras.Sequential([
        keras.layers.Flatten(input_shape=(28, 28, 1)),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


def build_basic_cnn_model():
    model = keras.Sequential([
        keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
        keras.layers.MaxPooling2D((2, 2)),

        keras.layers.Conv2D(64, (3, 3), activation="relu"),
        keras.layers.MaxPooling2D((2, 2)),

        keras.layers.Flatten(),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


def build_deeper_cnn_model():
    model = keras.Sequential([
        keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.25),

        keras.layers.Conv2D(64, (3, 3), activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.25),

        keras.layers.Conv2D(128, (3, 3), activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.30),

        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.50),

        keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# -----------------------------
# 4. Train Model
# -----------------------------
def train_model(model, model_name, x_train, y_train_cat, x_test, y_test_cat):
    print(f"\nTraining {model_name}...")

    checkpoint_path = os.path.join(MODEL_DIR, f"{model_name}.keras")

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        )
    ]

    history = model.fit(
        x_train,
        y_train_cat,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(x_test, y_test_cat),
        callbacks=callbacks,
        verbose=1
    )

    print(f"{model_name} training completed.")
    return history


# -----------------------------
# 5. Evaluate Model
# -----------------------------
def evaluate_model(model, model_name, x_test, y_test, y_test_cat):
    print(f"\nEvaluating {model_name}...")

    loss, accuracy = model.evaluate(x_test, y_test_cat, verbose=0)

    y_pred_prob = model.predict(x_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    print(f"{model_name} Accuracy: {accuracy:.4f}")
    print(f"{model_name} Loss: {loss:.4f}")

    report = classification_report(
        y_test,
        y_pred,
        target_names=CLASS_NAMES,
        output_dict=True
    )

    report_df = pd.DataFrame(report).transpose()
    report_path = os.path.join(RESULT_DIR, f"{model_name}_classification_report.csv")
    report_df.to_csv(report_path)

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(cmap="Blues", xticks_rotation=45)
    plt.title(f"{model_name} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, f"{model_name}_confusion_matrix.png"))
    plt.close()

    return {
        "Model": model_name,
        "Test Loss": loss,
        "Test Accuracy": accuracy
    }


# -----------------------------
# 6. Plot Training Curves
# -----------------------------
def plot_training_history(history, model_name):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history["accuracy"], label="Train Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{model_name} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, f"{model_name}_accuracy_curve.png"))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title(f"{model_name} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, f"{model_name}_loss_curve.png"))
    plt.close()


# -----------------------------
# 7. Show Sample Predictions
# -----------------------------
def save_sample_predictions(model, x_test, y_test):
    y_pred_prob = model.predict(x_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    indices = np.random.choice(len(x_test), 25, replace=False)

    plt.figure(figsize=(12, 12))

    for i, idx in enumerate(indices):
        plt.subplot(5, 5, i + 1)
        plt.imshow(x_test[idx].reshape(28, 28), cmap="gray")
        predicted = CLASS_NAMES[y_pred[idx]]
        actual = CLASS_NAMES[y_test[idx]]
        plt.title(f"P: {predicted}\nA: {actual}", fontsize=8)
        plt.axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, "sample_predictions.png"))
    plt.close()


# -----------------------------
# 8. Main Execution
# -----------------------------
def main():
    x_train, y_train, y_train_cat, x_test, y_test, y_test_cat = load_data()

    models = {
        "ANN_Model": build_ann_model(),
        "Basic_CNN_Model": build_basic_cnn_model(),
        "Deeper_CNN_Model": build_deeper_cnn_model()
    }

    results = []

    for model_name, model in models.items():
        model.summary()

        history = train_model(
            model,
            model_name,
            x_train,
            y_train_cat,
            x_test,
            y_test_cat
        )

        plot_training_history(history, model_name)

        result = evaluate_model(
            model,
            model_name,
            x_test,
            y_test,
            y_test_cat
        )

        results.append(result)

    results_df = pd.DataFrame(results)
    results_df.to_csv(os.path.join(RESULT_DIR, "model_comparison.csv"), index=False)

    best_model_name = results_df.sort_values(
        by="Test Accuracy",
        ascending=False
    ).iloc[0]["Model"]

    print("\nFinal Model Comparison:")
    print(results_df)

    print(f"\nBest Model: {best_model_name}")

    best_model = keras.models.load_model(os.path.join(MODEL_DIR, f"{best_model_name}.keras"))
    save_sample_predictions(best_model, x_test, y_test)

    print("\nProject completed successfully.")
    print("Models saved inside /models")
    print("Reports and charts saved inside /results")


if __name__ == "__main__":
    main()

Loading Fashion-MNIST dataset...
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training images: (60000, 28, 28, 1)
Testing images: (10000, 28, 28, 1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

...

 Non-trainable params: 704 (2.75 KB)


Training Deeper_CNN_Model...
Epoch 1/15
937/938 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.7071 - loss: 0.8518
Epoch 1: val_loss improved from None to 0.46516, saving model to models/Deeper_CNN_Model.keras

Epoch 1: finished saving model to models/Deeper_CNN_Model.keras
938/938 ━━━━━━━━━━━━━━━━━━━━ 102s 104ms/step - accuracy: 0.7830 - loss: 0.6092 - val_accuracy: 0.8222 - val_loss: 0.4652
Epoch 2/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.8499 - loss: 0.4134
Epoch 2: val_loss improved from 0.46516 to 0.34723, saving model to models/Deeper_CNN_Model.keras

Epoch 2: finished saving model to models/Deeper_CNN_Model.keras
938/938 ━━━━━━━━━━━━━━━━━━━━ 99s 105ms/step - accuracy: 0.8566 - loss: 0.3960 - val_accuracy: 0.8730 - val_loss: 0.3472
Epoch 3/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.8722 - loss: 0.3563
Epoch 3: val_loss improved from 0.34723 to 0.33001, saving model to models/Deeper_CNN_Model.keras

Epoch 3: finished saving model to models/Dee

<Figure size 1000x800 with 0 Axes>

<Figure size 1000x800 with 0 Axes>

<Figure size 1000x800 with 0 Axes>